# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [1]:
!pip install -qU "langchain>=1.0" "langchain-core>=0.3" "langchain-community>=0.4" "langchain-classic>=0.3" langchain-openai langchain-elasticsearch tiktoken jq lark elasticsearch


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_community.vectorstores.elasticsearch import ElasticsearchStore

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ELASTIC_URL = os.getenv("ELASTIC_URL")
ELASTIC_API_KEY = os.getenv("ELASTIC_API_KEY")

## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial.

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [3]:
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

INDEX_NAME = "workplace-docs"

vectorstore = ElasticsearchStore(
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
    index_name=INDEX_NAME,
    embedding=embeddings
)

C:\Users\sisaz\AppData\Local\Temp\ipykernel_20932\3024188395.py:5: LangChainPendingDeprecationWarning: The class `ElasticsearchStore` will be deprecated in a future version. Use `Use class in langchain-elasticsearch package` instead.
  vectorstore = ElasticsearchStore(


## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [4]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [5]:
from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datetime import datetime


def metadata_func(record: dict, metadata: dict) -> dict:
    metadata["name"] = record.get("name", "")
    metadata["summary"] = record.get("summary", "")
    metadata["url"] = record.get("url", "")
    metadata["category"] = record.get("category", "")
    updated_at = record.get("updated_at", "")
    try:
        metadata["updated_at"] = datetime.strptime(updated_at, "%Y-%m-%d").isoformat() if updated_at else datetime.now().isoformat()
    except ValueError:
        metadata["updated_at"] = datetime.now().isoformat()
    return metadata


loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000,
    chunk_overlap=200,
)

docs = loader.load_and_split(text_splitter=text_splitter)
print(f"Loaded {len(docs)} document chunks")
print("Sample metadata:", docs[0].metadata)

Loaded 15 document chunks
Sample metadata: {'source': 'C:\\Users\\sisaz\\PycharmProjects\\IronHackProjects\\Labs\\w7_d4_Labs\\lab-chatbot-with-multi-query-retriever\\temp.json', 'seq_num': 1, 'name': 'Work From Home Policy', 'summary': 'This policy outlines the guidelines for full-time remote work, including eligibility, equipment and resources, workspace requirements, communication expectations, performance expectations, time tracking and overtime, confidentiality and data security, health and well-being, and policy reviews and updates. Employees are encouraged to direct any questions or concerns', 'url': './sharepoint/Work from home policy.txt', 'category': 'teams', 'updated_at': '2020-03-01T00:00:00'}


### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [6]:
from datetime import datetime
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

clean_docs = []
for doc in docs:
    metadata = doc.metadata.copy()
    if metadata.get("updated_at") in ["", None, "null"]:
        metadata["updated_at"] = datetime.now().isoformat()
    clean_docs.append(doc.model_copy(update={"metadata": metadata}))

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

vectorstore = ElasticsearchStore.from_documents(
    clean_docs,
    embeddings,
    index_name=INDEX_NAME,
    es_url=ELASTIC_URL,
    es_api_key=ELASTIC_API_KEY,
)

llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=OPENAI_API_KEY
)

retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    llm=llm
)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [7]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain as lc_chain
import logging

# Enable detailed logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Multi-query generator with 3 variants
MULTI_QUERY_PROMPT = ChatPromptTemplate.from_template("""
Generate 3 diverse versions of this question for better retrieval. Vary phrasing, keywords, and perspectives:

{question}

Queries (one per line):
""")

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template("""
📄 [{source}]
{page_content}
---
""")

# Define the context prompt for the LLM
LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template("""
Answer the question based only on the following context:
{context}

Question: {question}
""")

def safe_combine_docs(docs):
    """Production-ready doc formatting with fallbacks"""
    doc_strings = []
    for i, doc in enumerate(docs):
        try:
            doc_dict = doc.model_dump()
            source = doc.metadata.get("name") or doc.metadata.get("source", f"Doc-{i}")
            doc_dict["source"] = source
            formatted = LLM_DOCUMENT_PROMPT.format(**doc_dict)
        except Exception as e:
            logger.warning(f"Doc format error: {e}")
            formatted = f"[Doc-{i}] {doc.page_content[:500]}..."
        doc_strings.append(formatted)
    return "\n\n".join(doc_strings)

# Self-healing chain: retry bad retrievals
def self_healing_retriever(query, max_tries=2):
    """Retry with rewritten query if empty results"""
    for attempt in range(max_tries):
        docs = retriever.invoke(query)
        if docs:
            return docs
        logger.info(f"Empty results (attempt {attempt+1}), rewriting...")
        query = llm.invoke(f"Rewrite for better retrieval: {query}").content
    return retriever.invoke(query)  # Fallback

_context = RunnableParallel(
    context=(RunnablePassthrough() | self_healing_retriever | safe_combine_docs),
    question=RunnablePassthrough(),
)

rag_chain = _context | LLM_CONTEXT_PROMPT | llm | StrOutputParser()

# Test with auto-multi-query
def multi_query_rag(question):
    """Generate + retrieve + answer"""

    query_chain = MULTI_QUERY_PROMPT | llm | StrOutputParser()

    generated_queries = query_chain.invoke({"question": question})

    print("\nGenerated Queries:")
    print("------------------")
    print(generated_queries)
    print("------------------\n")

    queries = [q.strip() for q in generated_queries.split("\n") if q.strip()]

    all_docs = []

    for q in queries:
        docs = self_healing_retriever(q)
        all_docs.extend(docs[:3])

    return rag_chain.invoke({
        "question": question,
        "context": safe_combine_docs(all_docs)
    })

print("---- Answer ----")

print(multi_query_rag("what is the nasa sales team?"))

---- Answer ----


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Generated Queries:
------------------
1. Can you provide information about the sales team at NASA?
2. What is the role of the sales team within NASA?
3. How does NASA's sales team operate and contribute to the organization's goals?
------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What details can you share about the sales department within NASA?', '2. Could you give me insights into the sales team operating within NASA?', "3. I'm interested in learning more about the sales team specifically within NASA. Can you provide relevant information?"]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-d4c7c9.es.europe-west1.gcp.elastic.cloud:443/workplace-docs/_search?_source_includes=metadata,text [status:200 duration:0.469s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-d4c7c9.es.europe-west1.gcp.elastic.cloud:443/workplace-docs/_search?_source_includes=metadata,text [status:200 duration:0.283s]
INF

The NASA sales team consists of two Area Vice-Presidents: Laura Martinez for North America and Gary Johnson for South America.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

## Iteration 1: Perspective-Based Multi-Query RAG

Instead of generic query paraphrasing, this iteration generates queries from **three distinct professional perspectives** — technical, managerial, and operational — to broaden the retrieval coverage and surface more relevant documents.

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

PERSPECTIVE_PROMPT = ChatPromptTemplate.from_template("""
You are an expert at reframing questions from different professional angles.
Given the question below, rewrite it from three perspectives:
1. Technical perspective (focus on systems, tools, processes)
2. Managerial perspective (focus on teams, strategy, decisions)
3. Operational perspective (focus on day-to-day tasks, workflows, people)

Original question: {question}

Output exactly 3 lines, one per perspective:
""")

def perspective_rag(question):
    query_chain = PERSPECTIVE_PROMPT | llm | StrOutputParser()
    generated = query_chain.invoke({"question": question})

    print("Perspective-Based Queries:")
    print("-" * 40)
    print(generated)
    print("-" * 40 + "\n")

    queries = [q.strip() for q in generated.strip().split("\n") if q.strip()]

    all_docs = []
    seen = set()
    for q in queries:
        for doc in retriever.invoke(q):
            key = doc.page_content[:100]
            if key not in seen:
                seen.add(key)
                all_docs.append(doc)

    context = safe_combine_docs(all_docs)

    answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the following context gathered from multiple perspectives:

{context}

Question: {question}

Provide a comprehensive answer that covers technical, managerial, and operational aspects where relevant.
""")

    chain = answer_prompt | llm | StrOutputParser()
    return chain.invoke({"context": context, "question": question})


print("---- Perspective-Based Answer ----")
print(perspective_rag("What are the vacation and time off policies?"))

---- Perspective-Based Answer ----


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Perspective-Based Queries:
----------------------------------------
1. Technical perspective: How is the vacation and time off policy integrated into the HR management system and payroll software?
2. Managerial perspective: How can we align the vacation and time off policies with our overall workforce management strategy and employee retention goals?
3. Operational perspective: How do employees request time off, track their vacation balances, and ensure coverage for their responsibilities during their absence?
----------------------------------------



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. How does the HR management system and payroll software incorporate the vacation and time off policy?', '2. In what way is the vacation and time off policy linked with the HR management system and payroll software?', '3. Can you explain the integration of the vacation and time off policy within the HR management system and payroll software?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-d4c7c9.es.europe-west1.gcp.elastic.cloud:443/workplace-docs/_search?_source_includes=metadata,text [status:200 duration:0.147s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-d4c7c9.es.europe-west1.gcp.elastic.cloud:443/workplace-

The vacation and time off policies at the company are designed to promote a healthy work-life balance and ensure that employees have the opportunity to rest and recharge. 

From a technical perspective, full-time employees accrue vacation time at a specific rate per month, which translates to a certain number of days per year. Part-time employees accrue vacation time on a pro-rata basis based on their scheduled work hours. Vacation time starts accruing from the first day of employment, but employees can only take vacation after completing their probationary period. Unused vacation time can be carried over to the next year up to a maximum limit, after which any additional unused time will be forfeited. 

Managerially, employees are required to submit vacation requests to their supervisor a certain number of weeks in advance, specifying the start and end dates of their vacation. Supervisors review and approve these requests based on business needs to ensure adequate coverage during the e

## Iteration 2: Iterative Refinement RAG

This iteration uses a **two-pass retrieval** strategy. In the first pass it retrieves an initial set of documents. It then asks the LLM to identify gaps and generate a refined follow-up query. The second pass retrieves additional documents to fill those gaps, and the final answer is grounded in the combined evidence.

In [9]:
GAP_PROMPT = ChatPromptTemplate.from_template("""
You retrieved the following documents to answer a question, but the answer may be incomplete.

Documents:
{context}

Original question: {question}

Identify what information is still missing and write ONE concise follow-up search query to find it.
Output only the query, nothing else.
""")

FINAL_ANSWER_PROMPT = ChatPromptTemplate.from_template("""
Use the following context from two rounds of retrieval to give a thorough answer.

Round 1 context:
{context_1}

Round 2 context (gap-filling):
{context_2}

Question: {question}

Answer:
""")

def iterative_refinement_rag(question):
    # Pass 1: initial retrieval
    docs_pass1 = retriever.invoke(question)
    context_1 = safe_combine_docs(docs_pass1)

    print("Pass 1 retrieved", len(docs_pass1), "documents")

    # Identify gaps and generate follow-up query
    gap_chain = GAP_PROMPT | llm | StrOutputParser()
    followup_query = gap_chain.invoke({"context": context_1, "question": question}).strip()

    print(f"\nFollow-up query: {followup_query}\n")

    # Pass 2: gap-filling retrieval
    docs_pass2 = retriever.invoke(followup_query)
    context_2 = safe_combine_docs(docs_pass2)

    print("Pass 2 retrieved", len(docs_pass2), "additional documents")

    # Final answer using both passes
    final_chain = FINAL_ANSWER_PROMPT | llm | StrOutputParser()
    return final_chain.invoke({
        "context_1": context_1,
        "context_2": context_2,
        "question": question
    })


print("---- Iterative Refinement Answer ----")
print(iterative_refinement_rag("What are the health benefits offered to employees?"))

---- Iterative Refinement Answer ----


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['What types of health benefits are available for employees?', 'What kind of health-related perks do employees receive?', 'What are the wellness advantages provided to the staff members?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-d4c7c9.es.europe-west1.gcp.elastic.cloud:443/workplace-docs/_search?_source_includes=metadata,text [status:200 duration:0.163s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-d4c7c9.es.europe-west1.gcp.elastic.cloud:443/workplace-docs/_search?_source_includes=metadata,text [status:200 duration:0.180s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
IN

Pass 1 retrieved 4 documents


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"



Follow-up query: health benefits offered to employees site:companywebsite.com



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:langchain_classic.retrievers.multi_query:Generated queries: ['1. What are the health benefits provided to employees on companywebsite.com?', '2. Can you list the employee health benefits available on companywebsite.com?', '3. What types of health benefits are offered to employees at companywebsite.com?']
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-d4c7c9.es.europe-west1.gcp.elastic.cloud:443/workplace-docs/_search?_source_includes=metadata,text [status:200 duration:0.156s]
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:elastic_transport.transport:POST https://my-elasticsearch-project-d4c7c9.es.europe-west1.gcp.elastic.cloud:443/workplace-docs/_search?_source_includes=metadata,text [status:200 duration:0.172s]
INFO:httpx:HTTP Request: POST 

Pass 2 retrieved 4 additional documents


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The health benefits offered to employees include health insurance, retirement plans, and paid time off. Employees are eligible for these benefits and will receive detailed information about the benefits package during orientation. To enroll in the benefits, employees need to carefully review the options, fill out the necessary forms, and submit them to the HR department within 30 days of their start date. Additionally, employees may designate beneficiaries for their life insurance and retirement plans if applicable.
